In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox

# =========================
# Machines (A/B/C naming shown in UI)
# =========================
DFA_STATES = ["A", "B"]          # A = even 0s, B = odd 0s
DFA_START  = "A"
DFA_ACCEPT = {"A"}
DFA_TRANS  = {"A": {"0": "B", "1": "A"},
              "B": {"0": "A", "1": "B"}}

NFA_STATES = ["A","B","C"]
NFA_START  = "A"
NFA_ACCEPT = {"C"}
NFA_TRANS  = {"A": {"a": {"A","B"}, "b": {"A"}},
              "B": {"b": {"C"}},
              "C": {}}

EPS = "ε"
NFAE_STATES = ["A","B","C","D","E"]
NFAE_START  = "A"
NFAE_ACCEPT = {"E"}
NFAE_TRANS  = {
    "A": {EPS: {"B"}},
    "B": {"a": {"B","C"}, "b": {"B"}},
    "C": {"b": {"D"}},
    "D": {"b": {"E"}},
    "E": {}
}

PDA_STATES = ["q_start","q_push","q_pop","q_accept"]
PDA_START  = "q_start"
PDA_ACCEPT = {"q_accept"}

TM_STATES = ["q0","q_find_b","q_find_c","q_check","q_accept","q_reject"]
TM_START  = "q0"
TM_ACCEPT = {"q_accept"}

# =========================
# Engines
# =========================
def dfa_run(s: str):
    st = DFA_START; zeros = 0; trace=[f"start: {st}"]; path=[st]
    for i,ch in enumerate(s,1):
        if ch not in "01":
            trace.append("invalid symbol (alphabet {0,1}) → REJECT")
            return False, trace, path, zeros
        if ch=="0": zeros += 1
        nxt = DFA_TRANS[st][ch]
        trace.append(f"step {i}: read {ch}, {st} → {nxt}")
        st = nxt; path.append(st)
    ok = st in DFA_ACCEPT
    trace.append(f"final: {st} (zeros={zeros}, parity={'even' if zeros%2==0 else 'odd'}) → {'ACCEPT' if ok else 'REJECT'}")
    return ok, trace, path, zeros

def nfa_run(s: str):
    active={NFA_START}; steps=[active.copy()]; trace=[f"start: {sorted(active)}"]
    for i,ch in enumerate(s,1):
        if ch not in "ab":
            trace.append("invalid symbol (alphabet {a,b}) → REJECT"); return False, trace, steps
        nxt=set()
        for q in active: nxt |= NFA_TRANS.get(q,{}).get(ch,set())
        active=nxt; steps.append(active.copy())
        trace.append(f"step {i}: read {ch} → {sorted(active)}")
    ok = bool(active & NFA_ACCEPT)
    trace.append(f"final: {sorted(active)} → {'ACCEPT' if ok else 'REJECT'}")
    return ok, trace, steps

def nfae_eclose(S:set[str]):
    stack=list(S); seen=set(S)
    while stack:
        q=stack.pop()
        for nq in NFAE_TRANS.get(q,{}).get(EPS,set()):
            if nq not in seen: seen.add(nq); stack.append(nq)
    return seen

def nfae_run(s:str):
    active = nfae_eclose({NFAE_START}); steps=[active.copy()]
    trace=[f"start ε-closure: {sorted(active)}"]
    for i,ch in enumerate(s,1):
        if ch not in "ab":
            trace.append("invalid symbol (alphabet {a,b}) → REJECT"); return False, trace, steps
        move=set()
        for q in active: move |= NFAE_TRANS.get(q,{}).get(ch,set())
        active = nfae_eclose(move); steps.append(active.copy())
        trace.append(f"step {i}: read {ch} → move {sorted(move)} → ε-closure {sorted(active)}")
    ok = bool(active & NFAE_ACCEPT)
    trace.append(f"final ε-closure: {sorted(active)} → {'ACCEPT' if ok else 'REJECT'}")
    return ok, trace, steps

def pda_run(s:str):
    trace=["start: q_start"]; stack=[]; phase="push"
    if s=="": trace.append("ε input, stack empty → q_accept → ACCEPT"); return True, trace
    for i,ch in enumerate(s,1):
        if ch not in "ab": trace.append(f"invalid '{ch}' → REJECT"); return False, trace
        if ch=="a" and phase=="push":
            stack.append("A"); trace.append(f"step {i}: read a, push A → stack={len(stack)}")
        elif ch=="b":
            if phase=="push": phase="pop"; trace.append(f"step {i}: first b → switch to pop phase")
            if not stack: trace.append(f"step {i}: pop on empty stack → REJECT"); return False, trace
            stack.pop(); trace.append(f"step {i}: read b, pop A → stack={len(stack)}")
        else:
            trace.append(f"step {i}: saw a after b-phase → REJECT"); return False, trace
    ok = (len(stack)==0)
    trace.append("end: stack empty → q_accept → ACCEPT" if ok else "end: stack not empty → REJECT")
    return ok, trace

def tm_run(s:str):
    tape=list(s); trace=[f"start: tape={''.join(tape) if tape else 'ε'}"]
    if not tape: trace.append("n≥1 required → REJECT"); return False, trace
    def find_from(i, sym):
        for k in range(i, len(tape)):
            if tape[k]==sym: return k
        return -1
    a_idx=0; matched=0
    while True:
        i=find_from(a_idx,'a'); 
        if i==-1: break
        tape[i]='X'; trace.append(f"mark a at {i} → X | {''.join(tape)}")
        j=find_from(i+1,'b'); 
        if j==-1: trace.append("no b after a → REJECT"); return False, trace
        tape[j]='Y'; trace.append(f"mark b at {j} → Y | {''.join(tape)}")
        k=find_from(j+1,'c'); 
        if k==-1: trace.append("no c after b → REJECT"); return False, trace
        tape[k]='Z'; trace.append(f"mark c at {k} → Z | {''.join(tape)}")
        matched += 1; a_idx=i+1
    if matched==0: trace.append("no a found → REJECT"); return False, trace
    if any(ch not in "XYZ" for ch in tape): trace.append("unmarked symbol remains → REJECT"); return False, trace
    ok = tape.count('X')==tape.count('Y')==tape.count('Z')
    trace.append("all marked & counts equal → ACCEPT" if ok else "counts unequal → REJECT")
    return ok, trace

# =========================
# UI (Mint style)
# =========================
PAL = {
    "bg": "#efeaf7",
    "panel": "#f7f5fb",
    "panel_border": "#d6d1e3",
    "text": "#1a1b20",
    "muted": "#5f6070",
    "accent": "#2fa87a",
    "ok": "#1aaa66",
    "err": "#e15252",
    "canvas": "#ffffff",
    "node": {
        "DFA":  ("#6ac07b", "#a6e6b5"),
        "NFA":  ("#6ac07b", "#a6e6b5"),
        "NFAE": ("#6ac07b", "#a6e6b5"),
        "PDA":  ("#6ac07b", "#a6e6b5"),
        "TM":   ("#6ac07b", "#a6e6b5"),
    },
    "th_state":  "#b6e3c8",
    "th_col0":   "#f9ef9b",
    "th_col1":   "#a8c3ff",
    "th_col2":   "#ffd8a8",
    "row_even":  "#ffffff",
    "row_odd":   "#f3f0fa",
}
EDGE_COLOR = "#101418"            # very dark for contrast
ARROW = (16, 20, 8)               # bigger heads

class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Automata Playground – DFA / NFA / NFAε / PDA / TM")
        self.geometry("1280x860"); self.minsize(1120, 780)

        self.method = tk.StringVar(value="DFA (even 0s)")
        self.input_str = tk.StringVar()
        self._R = 42

        # last-run highlights
        self._dfa_last_final=None; self._dfa_last_ok=None
        self._nfa_last_set=set(); self._nfa_last_ok=False
        self._nfae_last_set=set(); self._nfae_last_ok=False
        self._pda_last_ok=None; self._tm_last_ok=None

        self._style()
        self._build_top()
        self._build_body()
        self._load_method()

        self.canvas.bind("<Configure>", lambda e: self._redraw())
        self.bind("<Configure>", lambda e: self._wrap_meta())

    # ------- style -------
    def _style(self):
        self.configure(bg=PAL["bg"])
        s = ttk.Style(); s.theme_use("default")
        s.configure("TFrame", background=PAL["bg"])
        s.configure("TLabel", background=PAL["bg"], foreground=PAL["text"])
        s.configure("Header.TLabel", background=PAL["panel"], foreground=PAL["text"], font=("Segoe UI", 11, "bold"))
        s.configure("TButton", background=PAL["panel"], foreground=PAL["text"])
        s.configure("Accent.TButton", background=PAL["accent"], foreground="#ffffff")
        s.map("Accent.TButton", background=[("active", PAL["accent"])])
        s.configure("TEntry", fieldbackground="#ffffff", foreground=PAL["text"])
        s.configure("TCombobox", fieldbackground="#ffffff", foreground=PAL["text"])
        s.configure("Treeview", background=PAL["row_even"], fieldbackground=PAL["row_even"],
                    foreground=PAL["text"], bordercolor=PAL["panel_border"])
        s.configure("Treeview.Heading", font=("Segoe UI", 10, "bold"), foreground=PAL["text"])

    # ------- header -------
    def _build_top(self):
        bar = ttk.Frame(self, padding=(12,10)); bar.pack(side=tk.TOP, fill=tk.X)
        row = ttk.Frame(bar); row.pack(fill=tk.X)

        ttk.Label(row, text="Method:").grid(row=0, column=0, sticky="w", padx=(0,6))
        cb = ttk.Combobox(row, textvariable=self.method,
                          values=["DFA (even 0s)","NFA (ends with 'ab')","NFAε ((a|b)*abb)","PDA (a^n b^n)","TM (a^n b^n c^n)"],
                          state="readonly", width=26)
        cb.grid(row=0, column=1, sticky="w", padx=(0,12))
        cb.bind("<<ComboboxSelected>>", lambda e: self._load_method())

        ttk.Label(row, text="INPUT THE STRING HERE").grid(row=0, column=2, sticky="w")
        ttk.Entry(row, textvariable=self.input_str).grid(row=0, column=3, sticky="ew", padx=(8,10))

        ttk.Button(row, text="Run", command=self.on_run).grid(row=0, column=4, padx=4)
        ttk.Button(row, text="Clear", command=self.on_clear).grid(row=0, column=5, padx=4)

        self.result_lbl = ttk.Label(row, text="Result: —", font=("Segoe UI", 12, "bold"))
        self.result_lbl.grid(row=0, column=6, padx=(16,8), sticky="w")

        ttk.Button(row, text="Open Diagram (Large)", command=self.open_large, style="Accent.TButton")\
            .grid(row=0, column=7, padx=(8,0), sticky="e")
        row.grid_columnconfigure(3, weight=1)

        line2 = ttk.Frame(bar); line2.pack(fill=tk.X, pady=(6,0))
        self.meta = ttk.Label(line2, text="", wraplength=10, foreground=PAL["muted"])
        self.meta.pack(side=tk.LEFT, anchor="w")

    def _wrap_meta(self):
        try:
            self.meta.configure(wraplength=max(self.winfo_width()-40, 300))
        except Exception:
            pass

    # ------- body -------
    def _build_body(self):
        main = ttk.Frame(self); main.pack(fill=tk.BOTH, expand=True)
        left = ttk.Frame(main, padding=(12,10)); left.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        right = ttk.Frame(main, padding=(12,10)); right.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        ttk.Label(left, text="Transition Table / Actions", style="Header.TLabel").pack(anchor="w")
        self.tree = ttk.Treeview(left, columns=(), show="headings", height=9)
        self.tree.pack(fill=tk.X, pady=(6,12))

        ttk.Label(left, text="Step-by-step Trace", style="Header.TLabel").pack(anchor="w")
        self.trace = tk.Text(left, height=28, wrap="none", bg=PAL["row_odd"], fg=PAL["text"], insertbackground=PAL["text"])
        self.trace.configure(font=("Consolas", 11))
        self.trace.pack(fill=tk.BOTH, expand=True, pady=(6,0))

        ttk.Label(right, text="State Diagram", style="Header.TLabel").pack(anchor="w")
        self.canvas = tk.Canvas(right, bg=PAL["canvas"], highlightthickness=1, highlightbackground=PAL["panel_border"])
        self.canvas.pack(fill=tk.BOTH, expand=True, pady=(6,0))

        self.samples = {
            "DFA (even 0s)": ["ε (empty allowed)","0010","010","1100","111","000","1010"],
            "NFA (ends with 'ab')": ["babab","baa","ab","aab","baba","aaab"],
            "NFAε ((a|b)*abb)": ["abb","aabb","bababb","ab","aab","baba"],
            "PDA (a^n b^n)": ["","aaabbb","aab","ab","aaaabbbb","aabbab","bbaa"],
            "TM (a^n b^n c^n)": ["aaabbbccc","aabbcc","abc","aabbbccc","aaaabbbbcccc","aaabbbc"],
        }

    # ------- load method -------
    def _load_method(self):
        m = self.method.get()
        self.result_lbl.config(text="Result: —", foreground=PAL["text"])

        if m.startswith("DFA"):
            self.meta.config(text="(Alphabet: {0,1}; ε accepted)")
            cols=("State","on 0","on 1"); self.tree.config(columns=cols)
            self._color_headers(cols)
            rows=[(q,DFA_TRANS[q]["0"],DFA_TRANS[q]["1"]) for q in DFA_STATES]
            self._fill_rows(rows)
        elif "NFAε" in m:
            self.meta.config(text="(Alphabet: {a,b}, with ε-transitions)")
            cols=("State","on a","on b","on ε"); self.tree.config(columns=cols)
            self._color_headers(cols)
            def fmt(S): return "{" + ",".join(sorted(S)) + "}" if S else "∅"
            rows=[]
            for q in NFAE_STATES:
                row=NFAE_TRANS.get(q,{})
                rows.append((q, fmt(row.get("a",set())), fmt(row.get("b",set())), fmt(row.get(EPS,set()))))
            self._fill_rows(rows)
        elif m.startswith("NFA"):
            self.meta.config(text="(Alphabet: {a,b})")
            cols=("State","on a","on b"); self.tree.config(columns=cols)
            self._color_headers(cols)
            def fmt(S): return "{" + ",".join(sorted(S)) + "}" if S else "∅"
            rows=[]
            for q in NFA_STATES:
                row=NFA_TRANS.get(q,{})
                rows.append((q, fmt(row.get("a",set())), fmt(row.get("b",set()))))
            self._fill_rows(rows)
        elif m.startswith("PDA"):
            self.meta.config(text="(Alphabet: {a,b}, push A on a, pop on b; ε accepted)")
            cols=("From","Input/Stack Action","To"); self.tree.config(columns=cols)
            self._color_headers(cols)
            rows=[
                ("q_start","ε → go push mode","q_push"),
                ("q_push","read a, push A","q_push"),
                ("q_push","read b, pop A","q_pop"),
                ("q_pop", "read b, pop A","q_pop"),
                ("q_pop", "stack empty & ε","q_accept"),
            ]
            self._fill_rows(rows)
        else:
            self.meta.config(text="(Alphabet: {a,b,c}, n≥1; marking a→X, b→Y, c→Z)")
            cols=("From","Action","To"); self.tree.config(columns=cols)
            self._color_headers(cols)
            rows=[
                ("q0","mark a→X; move right","q_find_b"),
                ("q_find_b","find next b; mark→Y","q_find_c"),
                ("q_find_c","find next c; mark→Z","q0 (next a)"),
                ("q0","no more a","q_check"),
                ("q_check","all X,Y,Z equal","q_accept"),
                ("q_check","mismatch","q_reject"),
            ]
            self._fill_rows(rows)

        # reset highlights & sample
        self._dfa_last_final=None; self._dfa_last_ok=None
        self._nfa_last_set=set(); self._nfa_last_ok=False
        self._nfae_last_set=set(); self._nfae_last_ok=False
        self._pda_last_ok=None; self._tm_last_ok=None
        self.input_str.set(self.samples[m][0])
        self._redraw()

    def _color_headers(self, cols):
        for i,c in enumerate(cols):
            self.tree.heading(c, text=c)
            self.tree.column(c, anchor=tk.CENTER, width=160)
        if hasattr(self, "_th_bar"): self._th_bar.destroy()
        self._th_bar = tk.Canvas(self.tree.master, height=6, bg=PAL["panel"], highlightthickness=0)
        self._th_bar.pack(fill=tk.X, before=self.tree)
        self._th_bar.after(50, self._paint_header_colors)

    def _paint_header_colors(self):
        self._th_bar.delete("all")
        cols = self.tree["columns"]
        x = 0
        for idx,c in enumerate(cols):
            w = self.tree.column(c, "width")
            color = PAL["th_state"] if idx==0 else PAL["th_col0"] if idx==1 else PAL["th_col1"] if idx==2 else PAL["th_col2"]
            self._th_bar.create_rectangle(x,0,x+w,6, fill=color, outline=color)
            x += w

    def _fill_rows(self, rows):
        for r in self.tree.get_children(): self.tree.delete(r)
        for i,row in enumerate(rows):
            iid = self.tree.insert("", tk.END, values=row)
            self.tree.item(iid, tags=("odd" if i%2 else "even",))
        self.tree.tag_configure("even", background=PAL["row_even"])
        self.tree.tag_configure("odd", background=PAL["row_odd"])

    # ------- actions -------
    def on_clear(self):
        self.input_str.set("")
        self.trace.delete("1.0", tk.END)
        self.result_lbl.config(text="Result: —", foreground=PAL["text"])
        self._dfa_last_final=None; self._dfa_last_ok=None
        self._nfa_last_set=set(); self._nfa_last_ok=False
        self._nfae_last_set=set(); self._nfae_last_ok=False
        self._pda_last_ok=None; self._tm_last_ok=None
        self._redraw()

    def on_run(self):
        s = self.input_str.get(); m=self.method.get()

        if m.startswith("DFA"):
            if any(ch not in "01" for ch in s): messagebox.showerror("Invalid Input","Use only 0 or 1."); return
            ok,trace,path,zeros = dfa_run(s)
            self.result_lbl.config(text=f"Result: {'ACCEPT' if ok else 'REJECT'}",
                                   foreground=PAL["ok"] if ok else PAL["err"])
            self.meta.config(text=f"Zeros: {zeros} | Parity: {'even' if zeros%2==0 else 'odd'}")
            self.trace.delete("1.0",tk.END); self.trace.insert(tk.END,"\n".join(trace))
            self._dfa_last_final = path[-1]; self._dfa_last_ok = ok

        elif "NFAε" in m:
            if not s or any(ch not in "ab" for ch in s): messagebox.showerror("Invalid Input","Alphabet {a,b}; input non-empty."); return
            ok,trace,steps=nfae_run(s)
            self.result_lbl.config(text=f"Result: {'ACCEPT' if ok else 'REJECT'}",
                                   foreground=PAL["ok"] if ok else PAL["err"])
            self.meta.config(text=f"Final ε-closure: {sorted(steps[-1])}")
            self.trace.delete("1.0",tk.END); self.trace.insert(tk.END,"\n".join(trace))
            self._nfae_last_set=steps[-1]; self._nfae_last_ok=ok

        elif m.startswith("NFA"):
            if not s or any(ch not in "ab" for ch in s): messagebox.showerror("Invalid Input","Alphabet {a,b}; input non-empty."); return
            ok,trace,steps=nfa_run(s)
            self.result_lbl.config(text=f"Result: {'ACCEPT' if ok else 'REJECT'}",
                                   foreground=PAL["ok"] if ok else PAL["err"])
            self.meta.config(text=f"Final active set: {sorted(steps[-1])}")
            self.trace.delete("1.0",tk.END); self.trace.insert(tk.END,"\n".join(trace))
            self._nfa_last_set=steps[-1]; self._nfa_last_ok=ok

        elif m.startswith("PDA"):
            if any(ch not in "ab" for ch in s): messagebox.showerror("Invalid Input","Alphabet {a,b}."); return
            ok,trace=pda_run(s)
            self.result_lbl.config(text=f"Result: {'ACCEPT' if ok else 'REJECT'}",
                                   foreground=PAL["ok"] if ok else PAL["err"])
            self.meta.config(text=f"Stack empty at end: {'yes' if ok else 'no'}")
            self.trace.delete("1.0",tk.END); self.trace.insert(tk.END,"\n".join(trace))
            self._pda_last_ok=ok

        else:  # TM
            if not s or any(ch not in "abc" for ch in s): messagebox.showerror("Invalid Input","Alphabet {a,b,c}; n≥1."); return
            ok,trace=tm_run(s)
            self.result_lbl.config(text=f"Result: {'ACCEPT' if ok else 'REJECT'}",
                                   foreground=PAL["ok"] if ok else PAL["err"])
            self.meta.config(text=f"Marked counts equal? {'yes' if ok else 'no'}")
            self.trace.delete("1.0",tk.END); self.trace.insert(tk.END,"\n".join(trace))
            self._tm_last_ok=ok

        self._redraw()

    # ------- large diagram -------
    def open_large(self):
        m=self.method.get()
        title = ("NFAε Diagram ((a|b)*abb)" if "NFAε" in m else
                 "PDA Diagram (a^n b^n)" if "PDA" in m else
                 "TM Diagram (a^n b^n c^n)" if "TM" in m else
                 "Diagram")
        win = tk.Toplevel(self); win.title(title); win.geometry("1320x820"); win.minsize(1000,600)
        tk.Label(win, text=title, bg=PAL["bg"], fg=PAL["text"], font=("Segoe UI", 12, "bold")).pack(anchor="w", padx=10, pady=8)
        big = tk.Canvas(win, bg=PAL["canvas"], highlightthickness=1, highlightbackground=PAL["panel_border"])
        big.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        def redraw(_=None):
            old_canvas, old_R = self.canvas, self._R
            try:
                self.canvas = big; self._redraw(scale_for_big=True)
            finally:
                self.canvas = old_canvas; self._R = old_R
        big.bind("<Configure>", redraw); redraw()

    # =========================
    # Drawing helpers (edges first → nodes → halos)
    # =========================
    def _node_core(self, x,y,name,kind,accept=False,start=False):
        r=self._R
        inner, _halo = PAL["node"][kind]
        self.canvas.create_oval(x-r, y-r, x+r, y+r, outline="#1d1d1d", width=2.8, fill=inner, tags=("node",))
        if accept:
            self.canvas.create_oval(x-(r-6), y-(r-6), x+(r-6), y+(r-6), outline="#1d1d1d", width=2.6, tags=("node",))
        if start:
            self.canvas.create_line(x-120, y, x-r-3, y, width=4, arrow=tk.LAST, arrowshape=ARROW, fill=EDGE_COLOR, tags=("start",))
        self.canvas.create_text(x, y, text=name, font=("Segoe UI", 12, "bold"), fill="#0b270f", tags=("node_text",))

    def _halo(self, x,y,color):
        r=self._R
        self.canvas.create_oval(x-(r+12), y-(r+12), x+(r+12), y+(r+12),
                                outline=color, width=6, tags=("halo",))

    def _curved(self, p_src, p_dst, label, up=True):
        r=self._R; x1,y1=p_src; x2,y2=p_dst
        dx,dy=x2-x1,y2-y1; L=max((dx*dx+dy*dy)**0.5,1e-6); ux,uy=dx/L,dy/L
        sx,sy=x1+ux*(r+3), y1+uy*(r+3); ex,ey=x2-ux*(r+3), y2-uy*(r+3)
        px,py=-uy,ux; curve=0.35*L*(1 if up else -1)
        cx,cy=(sx+ex)/2 + px*curve/2, (sy+ey)/2 + py*curve/2
        self.canvas.create_line(sx,sy,cx,cy,ex,ey, smooth=True, splinesteps=48,
                                width=4, fill=EDGE_COLOR, arrow=tk.LAST, arrowshape=ARROW,
                                capstyle=tk.ROUND, tags=("edge",))
        lx,ly=(sx+2*cx+ex)/4, (sy+2*cy+ey)/4
        self.canvas.create_text(lx, ly-(16 if up else -16), text=label, font=("Segoe UI", 12), fill=EDGE_COLOR,
                                tags=("edge_label",))

    def _straight(self, p_src, p_dst, label):
        r=self._R; x1,y1=p_src; x2,y2=p_dst
        dx,dy=x2-x1,y2-y1; L=max((dx*dx+dy*dy)**0.5,1e-6); ux,uy=dx/L,dy/L
        sx,sy=x1+ux*(r+3), y1+uy*(r+3); ex,ey=x2-ux*(r+3), y2-uy*(r+3)
        self.canvas.create_line(sx,sy,ex,ey, width=4, fill=EDGE_COLOR, arrow=tk.LAST, arrowshape=ARROW,
                                capstyle=tk.ROUND, tags=("edge",))
        mx,my=(sx+ex)/2,(sy+ey)/2
        self.canvas.create_text(mx, my-16, text=label, font=("Segoe UI", 12), fill=EDGE_COLOR, tags=("edge_label",))

    def _loop(self, center, label):
        r=self._R; x,y=center
        bbox=(x-r, y-r-1.7*r, x+r, y+r-1.7*r)
        self.canvas.create_arc(*bbox, start=210, extent=240, style=tk.ARC, width=4,
                               outline=EDGE_COLOR, tags=("edge",))
        tip_x=x+r*0.88; tip_y=y-0.62*r
        self.canvas.create_line(tip_x-0.42*r, tip_y-0.32*r, tip_x, tip_y,
                                width=4, fill=EDGE_COLOR, arrow=tk.LAST, arrowshape=ARROW,
                                capstyle=tk.ROUND, tags=("edge",))
        self.canvas.create_text(x, y-2.05*r, text=label, font=("Segoe UI", 12),
                                fill=EDGE_COLOR, tags=("edge_label",))

    def _eps_edge(self, p_src, p_dst):
        r=self._R; x1,y1=p_src; x2,y2=p_dst
        dx,dy=x2-x1,y2-y1; L=max((dx*dx+dy*dy)**0.5,1e-6); ux,uy=dx/L,dy/L
        sx,sy=x1+ux*(r+3), y1+uy*(r+3); ex,ey=x2-ux*(r+3), y2-uy*(r+3)
        px,py=-uy,ux; curve=0.70*L
        cx,cy=(sx+ex)/2 + px*curve/2, (sy+ey)/2 + py*curve/2
        self.canvas.create_line(sx,sy,cx,cy,ex,ey, smooth=True, splinesteps=48,
                                width=4, dash=(6,6), fill=EDGE_COLOR,
                                arrow=tk.LAST, arrowshape=ARROW,
                                capstyle=tk.ROUND, tags=("edge",))
        lx,ly=(sx+2*cx+ex)/4, (sy+2*cy+ey)/4
        self.canvas.create_text(lx, ly-0.4*r, text=EPS, font=("Segoe UI", 12), fill=EDGE_COLOR, tags=("edge_label",))

    # ------- draw per machine (edges → nodes → halos + z-order) -------
    def _draw_dfa(self, big=False):
        cv=self.canvas; W=max(cv.winfo_width(),300); H=max(cv.winfo_height(),260); M=self._R*1.0
        xA=M+(W-2*M)*0.28; xB=M+(W-2*M)*0.72; y=M+(H-2*M)*0.62
        # edges first
        self._curved((xA,y),(xB,y),"0",up=True)
        self._curved((xB,y),(xA,y),"0",up=False)
        self._loop((xA,y),"1"); self._loop((xB,y),"1")
        # nodes
        self._node_core(xA,y,"A","DFA",accept=("A" in DFA_ACCEPT),start=True)
        self._node_core(xB,y,"B","DFA",accept=("B" in DFA_ACCEPT))
        # halos after (if last run)
        if self._dfa_last_final:
            color = PAL["ok"] if self._dfa_last_ok else PAL["err"]
            x,y = (xA,y) if self._dfa_last_final=="A" else (xB,y)
            self._halo(x,y,color)
        # bring edges on top of fills, but text above everything
        cv.tag_lower("node"); cv.tag_raise("edge"); cv.tag_raise("edge_label"); cv.tag_raise("halo"); cv.tag_raise("node_text"); cv.tag_raise("start")

    def _draw_nfa(self, big=False):
        cv=self.canvas; W=max(cv.winfo_width(),300); H=max(cv.winfo_height(),260); M=self._R*1.0
        span=(W-2*M); y=M+(H-2*M)*0.62
        xA=M+span*0.22; xB=M+span*0.50; xC=M+span*0.80
        # edges
        self._loop((xA,y),"a,b")
        self._straight((xA,y),(xB,y),"a")
        self._straight((xB,y),(xC,y),"b")
        # nodes
        self._node_core(xA,y,"A","NFA",start=True)
        self._node_core(xB,y,"B","NFA")
        self._node_core(xC,y,"C","NFA",accept=True)
        # halos for current set
        final = self._nfa_last_set or set()
        for q,(x,y) in [("A",(xA,y)),("B",(xB,y)),("C",(xC,y))]:
            if q in final:
                color = PAL["ok"] if (self._nfa_last_ok and q in NFA_ACCEPT) else PAL["err"]
                self._halo(x,y,color)
        cv.tag_lower("node"); cv.tag_raise("edge"); cv.tag_raise("edge_label"); cv.tag_raise("halo"); cv.tag_raise("node_text"); cv.tag_raise("start")

    def _draw_nfae(self, big=False):
        cv=self.canvas; W=max(cv.winfo_width(),300); H=max(cv.winfo_height(),260); M=self._R*1.0
        span=(W-2*M); y=M+(H-2*M)*0.62
        xA=M+span*(0.12 if big else 0.12)
        xB=M+span*(0.30)
        xC=M+span*(0.52)
        xD=M+span*(0.74)
        xE=M+span*(0.90)
        yC=y-32; yD=y+32
        # edges
        self._eps_edge((xA,y),(xB,y))
        self._loop((xB,y),"a,b")
        self._straight((xB,y),(xC,yC),"a")
        self._straight((xC,yC),(xD,yD),"b")
        self._straight((xD,yD),(xE,y),"b")
        # nodes
        self._node_core(xA,y,"A","NFAE",start=True)
        self._node_core(xB,y,"B","NFAE")
        self._node_core(xC,yC,"C","NFAE")
        self._node_core(xD,yD,"D","NFAE")
        self._node_core(xE,y,"E","NFAE",accept=True)
        # halos
        final = self._nfae_last_set or set()
        for q,(x,y) in [("A",(xA,y)),("B",(xB,y)),("C",(xC,yC)),("D",(xD,yD)),("E",(xE,y))]:
            if q in final:
                color = PAL["ok"] if (self._nfae_last_ok and q in NFAE_ACCEPT) else PAL["err"]
                self._halo(x,y,color)
        cv.tag_lower("node"); cv.tag_raise("edge"); cv.tag_raise("edge_label"); cv.tag_raise("halo"); cv.tag_raise("node_text"); cv.tag_raise("start")

    def _draw_pda(self, big=False):
        cv=self.canvas; W=max(cv.winfo_width(),300); H=max(cv.winfo_height(),260); M=self._R*1.0
        span=(W-2*M); y=M+(H-2*M)*0.62
        x0=M+span*(0.20); x1=M+span*(0.44); x2=M+span*(0.68); x3=M+span*(0.88)
        # edges
        self._straight((x0,y),(x1,y),"ε")
        self._loop((x1,y),"read a / push A")
        self._straight((x1,y),(x2,y),"read b / pop A")
        self._loop((x2,y),"read b / pop A")
        self._straight((x2,y),(x3,y),"stack empty / ε")
        # nodes
        self._node_core(x0,y,"q_start","PDA",start=True)
        self._node_core(x1,y,"q_push","PDA")
        self._node_core(x2,y,"q_pop","PDA")
        self._node_core(x3,y,"q_accept","PDA",accept=True)
        # halo
        if self._pda_last_ok is not None:
            color = PAL["ok"] if self._pda_last_ok else PAL["err"]
            self._halo(x3,y,color)
        cv.tag_lower("node"); cv.tag_raise("edge"); cv.tag_raise("edge_label"); cv.tag_raise("halo"); cv.tag_raise("node_text"); cv.tag_raise("start")

    def _draw_tm(self, big=False):
        cv=self.canvas; W=max(cv.winfo_width(),300); H=max(cv.winfo_height(),260); M=self._R*1.0
        span=(W-2*M); y=M+(H-2*M)*0.62
        x0=M+span*(0.18); x1=M+span*(0.40); x2=M+span*(0.62); x3=M+span*(0.80); x4=M+span*(0.92)
        # edges
        self._straight((x0,y),(x1,y),"mark a→X")
        self._straight((x1,y),(x2,y),"find b→Y")
        self._straight((x2,y),(x0,y),"find c→Z")
        self._straight((x0,y),(x3,y),"no more a")
        self._straight((x3,y),(x4,y),"all marked & equal")
        # nodes
        self._node_core(x0,y,"q0","TM",start=True)
        self._node_core(x1,y,"q_find_b","TM")
        self._node_core(x2,y,"q_find_c","TM")
        self._node_core(x3,y,"q_check","TM")
        self._node_core(x4,y,"q_accept","TM",accept=True)
        # halo
        if self._tm_last_ok is not None:
            color = PAL["ok"] if self._tm_last_ok else PAL["err"]
            self._halo(x4,y,color)
        cv.tag_lower("node"); cv.tag_raise("edge"); cv.tag_raise("edge_label"); cv.tag_raise("halo"); cv.tag_raise("node_text"); cv.tag_raise("start")

    # ---------- redraw entry ----------
    def _redraw(self, scale_for_big=False):
        cv=self.canvas; cv.delete("all")
        W=max(cv.winfo_width(),320); H=max(cv.winfo_height(),260); base=min(W,H)
        self._R = max(28, int(base*(0.055 if scale_for_big else 0.05)))

        m=self.method.get()
        if m.startswith("DFA"): self._draw_dfa()
        elif "NFAε" in m: self._draw_nfae()
        elif m.startswith("NFA"): self._draw_nfa()
        elif m.startswith("PDA"): self._draw_pda()
        else: self._draw_tm()

def main():
    App().mainloop()

if __name__ == "__main__":
    main()
